# 02 Baseline smoke test (pretrained Whisper-small)

Kaggle setup before running: Accelerator = GPU, Internet = On, add the dataset `echoscript-clean`.

This notebook only tests the pipeline on 20 FLEURS clips and 20 of our validation clips.
The full baseline run comes after this works.

## 1. GPU check and locate the dataset

In [ ]:
import torch, os, glob
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

hits = glob.glob("/kaggle/input/echoscript-clean/**/manifest.csv", recursive=True)
print("manifest found at:", hits)
ROOT = os.path.dirname(hits[0])
print("ROOT =", ROOT, "| clips:", len(os.listdir(f"{ROOT}/clips")))


## 2. Install (run before anything imports `datasets`; if you already imported it, Restart session first)

Why `datasets<4`: I believe newer versions refuse FLEURS's loading script. Unconfirmed, so this is a guess we test.

In [ ]:
!pip install -q jiwer "datasets<4"


## 3. Load FLEURS Hindi test

In [ ]:
from datasets import load_dataset, Audio
fl = load_dataset("google/fleurs", "hi_in", split="test", trust_remote_code=True)
fl = fl.cast_column("audio", Audio(sampling_rate=16000))
print("FLEURS Hindi test utterances:", len(fl))
print("transcription    :", fl[0]["transcription"])
print("raw_transcription:", fl[0]["raw_transcription"])


## 4. Text normalizer, model, transcribe function

`eval_norm` is applied to BOTH reference and model output before WER, so punctuation alone is not counted as errors.

In [ ]:
import re, unicodedata, jiwer, pandas as pd, soundfile as sf
from transformers import WhisperProcessor, WhisperForConditionalGeneration

def eval_norm(t):
    t = unicodedata.normalize("NFC", t).replace("_", " ")
    t = re.sub(r"[,.।॥!?:;\-\"'“”‘’()]", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

device = "cuda"
proc = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device).half().eval()

def transcribe(arrays, bs=16):
    out = []
    for i in range(0, len(arrays), bs):
        feats = proc.feature_extractor(arrays[i:i+bs], sampling_rate=16000,
                                       return_tensors="pt").input_features.to(device).half()
        with torch.no_grad():
            ids = model.generate(feats, language="hi", task="transcribe")
        out += proc.batch_decode(ids, skip_special_tokens=True)
    return out


## 5. Smoke test on 20 FLEURS utterances

In [ ]:
sub = fl.select(range(20))
hyp = transcribe([a["array"] for a in sub["audio"]])
refs = sub["transcription"]
r = [eval_norm(x) for x in refs]; h = [eval_norm(x) for x in hyp]
print("WER on 20 FLEURS utterances:", round(jiwer.wer(r, h), 3))
for a, b in list(zip(r, h))[:5]:
    print("REF:", a); print("HYP:", b); print()


## 6. Smoke test on 20 of our validation clips

In [ ]:
man = pd.read_csv(f"{ROOT}/manifest.csv")
s = man[man.split == "val"].head(20)
arr = [sf.read(f"{ROOT}/clips/{c}")[0] for c in s.clip]
hyp2 = transcribe(arr)
r2 = [eval_norm(t) for t in s.text]; h2 = [eval_norm(t) for t in hyp2]
print("WER on 20 val clips:", round(jiwer.wer(r2, h2), 3))
for a, b in list(zip(r2, h2))[:5]:
    print("REF:", a); print("HYP:", b); print()
